In [ ]:
import pandas as pd
import sqlite3
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors





In [ ]:
# https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset
spotify_df = pd.read_csv("../dataset/spotify-tracks/dataset.csv")
spotify_df.info()

In [ ]:
spotify_df = spotify_df.rename(columns={
    "track_name": "title",
    "artists": "artist_name"
})
spotify_df.head()

In [ ]:
conn = sqlite3.connect("../dataset/taste-profile/track_metadata.db")

msd_meta = pd.read_sql("""
SELECT song_id, title, artist_name
FROM songs
""", conn)

In [ ]:

def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = text.strip()
    text = re.sub(r"\(.*?\)", "", text)
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"feat\.?.*", "", text)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

spotify_df["title_clean"] = spotify_df["title"].apply(clean_text)
spotify_df["artist_clean"] = spotify_df["artist_name"].apply(clean_text)

msd_meta["title_clean"] = msd_meta["title"].apply(clean_text)
msd_meta["artist_clean"] = msd_meta["artist_name"].apply(clean_text)

In [ ]:
clean_merged = msd_meta.merge(
    spotify_df,
    on=["title_clean", "artist_clean"],
    how="inner"
)

print("Clean Merged songs:", len(clean_merged))

In [ ]:
clean_merged.head()

In [ ]:
clean_merged.info()

In [ ]:
from rapidfuzz.fuzz import ratio, token_set_ratio

matches = []

spotify_group = spotify_df.groupby("title_clean")

for _, row in msd_meta.iterrows():

    title = row["title_clean"]
    artist = row["artist_clean"]

    if title not in spotify_group.groups:
        continue

    candidates = spotify_group.get_group(title)

    for _, cand in candidates.iterrows():

        score = token_set_ratio(artist, cand["artist_clean"])

        if score > 90:
            matches.append({
                "song_id": row["song_id"],
                "spotify_track_id": cand["track_id"],
                "score": score
            })

fuzzy_matches = pd.DataFrame(matches)
print("Fuzzy matched songs:", len(fuzzy_matches))

## Use exact clean match

# EDA

In [ ]:
clean_merged = clean_merged.drop_duplicates(subset=["song_id"])
clean_merged["song_id"].nunique()

In [ ]:
fuzzy_matches = fuzzy_matches.drop_duplicates(subset=["song_id"])
fuzzy_matches["song_id"].nunique()

#### Distribution of audio features

In [ ]:
audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

clean_merged[audio_features].hist(
    bins=30,
    figsize=(12,8)
)

plt.tight_layout()
plt.show()

#### Feature correlation

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(
    clean_merged[audio_features].corr(),
    cmap="coolwarm",
    annot=True
)

plt.title("Audio Feature Correlation")
plt.show()

#### Genre distribution

In [ ]:
clean_merged["track_genre"].value_counts().head(15).plot(
    kind="bar",
    figsize=(10,5)
)

plt.title("Top Genres")
plt.show()

#### Popularity distribution

In [ ]:
clean_merged["popularity"].hist(bins=30)
plt.title("Popularity Distribution")
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

X = clean_merged[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

clean_merged["pca1"] = X_pca[:,0]
clean_merged["pca2"] = X_pca[:,1]
clean_merged["pca3"] = X_pca[:,2]

import plotly.express as px

fig = px.scatter_3d(
    clean_merged,
    x="pca1",
    y="pca2",
    z="pca3",
    color="track_genre",
    opacity=0.6
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(
    width=1000,
    height=800
)
fig.show()

# Content-based recommendation

In [ ]:
nn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11
)

nn_model.fit(X_scaled)

### Song index mapping

In [ ]:
songid_to_idx = {
    sid: i for i, sid in enumerate(clean_merged["song_id"])
}

idx_to_song = {
    i: sid for i, sid in enumerate(clean_merged["song_id"])
}

### Recommendation function

In [ ]:
def recommend_similar_songs(song_id, N=10):

    idx = songid_to_idx[song_id]

    distances, indices = nn_model.kneighbors(
        X_scaled[idx].reshape(1, -1),
        n_neighbors=N+1
    )

    rec_idxs = indices[0][1:]
    rec_dist = distances[0][1:]

    rec_df = clean_merged.iloc[rec_idxs][
        ["song_id","title_x","artist_name_x","track_genre"]
    ].copy()

    rec_df["similarity"] = 1 - rec_dist

    return rec_df.sort_values("similarity", ascending=False)

### Example usage

In [ ]:
song_id = clean_merged.iloc[3]["song_id"]

recommend_similar_songs(song_id)

# Evaluation

In [ ]:
song = clean_merged.iloc[100]

print("Input Song:")
print(song["title_x"], "-", song["artist_name_x"], "-", song["track_genre"])

print("\nRecommended Songs:")

recommend_similar_songs(song["song_id"], N=5)

## Save model

In [ ]:
import joblib

joblib.dump(nn_model, "../models/content_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")
clean_merged.to_csv("../models/song_features.csv", index=False)